# AdaMask Colab Quickstart

Runs a fast sanity check first (small model + TinyStories, a few minutes on a T4) to confirm the masked-diffusion pipeline works before committing to the full-scale run in `main.py`.

Before running: **Runtime > Change runtime type > GPU**.

In [ ]:
!git clone https://github.com/Lenue01/AdaMask322.git
%cd AdaMask322
!pip install -q transformers datasets tqdm

## 1. Sanity check

Trains a tiny model (256 hidden, 8 layers) on TinyStories for 10 epochs × 1500 steps, then samples 4 sequences. TinyStories is simple, clean prose -- easier to judge learning progress from than wikitext's markup noise. All flags below are editable. Expect simple, sometimes repetitive sentences, not full coherence -- this step just confirms masking, training, sampling, and decoding work end to end.

In [ ]:
!python sanity_check.py \
  --dataset-name roneneldan/TinyStories \
  --dataset-config default \
  --split train \
  --context-length 128 \
  --hidden-size 256 \
  --heads 8 \
  --layers 8 \
  --steps 32 \
  --batch-size 32 \
  --num-epochs 10 \
  --steps-per-epoch 1500 \
  --warmup-steps 100 \
  --save-every-epochs 1 \
  --max-workers 2 \
  --difficulty-loss-scale 0.3

## 2. Full-scale training (optional, slow)

Uses the real config (hidden_size=1024, 16 layers, fineweb-edu). This downloads a large dataset and runs for a long time -- lower `--num-epochs`/`--steps-per-epoch` if you just want to test it. Peak LR and warmup steps auto-scale from `--hidden-size`/`--num-epochs`/`--steps-per-epoch` (override with `--lr`/`--warmup-steps`).

Checkpoints (model + optimizer + scaler + token-difficulty state + architecture) save as `masked_diffusion_epoch_N.pt` every `--save-every-epochs`. Colab VMs are ephemeral -- **run from a Drive-mounted folder** (below), or checkpoints won't survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/AdaMask322_runs', exist_ok=True)
%cd /content/drive/MyDrive/AdaMask322_runs

In [ ]:
!python /content/AdaMask322/main.py \
  --model-name roberta-base \
  --dataset-name HuggingFaceFW/fineweb-edu \
  --dataset-config sample-10BT \
  --batch-size 128 \
  --context-length 128 \
  --hidden-size 1024 \
  --heads 16 \
  --layers 16 \
  --steps 64 \
  --num-epochs 50 \
  --steps-per-epoch 8000 \
  --save-every-epochs 2 \
  --difficulty-loss-scale 0.3

### Resuming after a disconnect

Pass `--resume` with the path to the last checkpoint to continue training (optimizer/scaler state and epoch count included) instead of starting over. If your flags don't match what the checkpoint was originally trained with, training now prints a warning naming the mismatched fields instead of silently drifting.

In [ ]:
!python /content/AdaMask322/main.py --resume masked_diffusion_epoch_10.pt

## 3. Generate from a checkpoint

Checkpoints save their own architecture now, so `generate.py` picks up `hidden_size`/`heads`/`layers`/`steps`/`context_length` automatically -- no matching flags needed. (A checkpoint saved before this notebook has none saved; for one of those, pass those flags manually, matching whatever trained it.)

Generation can revise its own earlier guesses instead of freezing tokens the instant they're revealed: `--remask-threshold` sets how unconfident the model must get in an already-revealed token before it's erased and reconsidered; `--max-remask-frac` caps how much of the sequence can be remasked per step. Defaults (0.2 / 0.1) are untested starting points -- lower `--max-remask-frac` if output looks too volatile, raise `--remask-threshold` if it looks stuck on early mistakes.

In [ ]:
!python /content/AdaMask322/generate.py masked_diffusion_epoch_16.pt \
  --num-samples 10 \
  --temperature .9 \
  --remask-threshold 0.2 \
  --max-remask-frac 0.1